Raw data rarely contains all information a model needs.
We create new columns from existing ones to help the model to find patterns it couldn't otherwise detect.

Rule of thumb: 80% of model accuracy comes from good features, not from the choice of algorithm.

## Importing required libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import sklearn
import json
import warnings

In [2]:
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('C:/Users/HP/Downloads/edunet-IBM AIML project/retail_raw_data.csv', parse_dates=['date'])
df = df.sort_values(['stockcode','date']).reset_index(drop=True)

In [4]:
print('Loaded:', df.shape)
print(f'Demand range: {df.demand.min()} – {df.demand.max()}')
print(f'Zero-demand rows: {(df.demand==0).sum():,} ({(df.demand==0).mean()*100:.1f}%)')
print('Columns:', list(df.columns))

Loaded: (22420, 20)
Demand range: 0 – 80995
Zero-demand rows: 7,451 (33.2%)
Columns: ['date', 'stockcode', 'description', 'demand', 'price', 'revenue', 'num_invoices', 'num_customers', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'quarter', 'week', 'country', 'lost_sales', 'stock_level', 'is_promotion', 'discount_pct']


## Time-Based Features

WHY CYCLICAL ENCODING (sin/cos)?
If month = 1,2,3,...,12 the model treats December(12) and January(1) as far apart. But seasonally they're neighbors.

Sin/cos wraps the calendar into a circle so the model, understands that Dec → Jan is a small step, not a big jump.

In [5]:
print('\n[1/6] Time features...')

df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['day']         = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['week_of_year']= df['date'].dt.isocalendar().week.astype(int)
df['quarter']     = df['date'].dt.quarter
df['day_of_year'] = df['date'].dt.dayofyear



[1/6] Time features...


In [6]:
# Cyclical encoding so Dec(12) and Jan(1) are "close" to the model
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)

In [7]:
# Holiday/event flags (UK retail)
df['is_christmas_season'] = df['month'].isin([11,12]).astype(int)
df['is_q4']               = (df['quarter'] == 4).astype(int)
df['is_january_dip']      = (df['month'] == 1).astype(int)

In [8]:
print('14 time features added')

14 time features added


## Lag Features (MOST IMPORTANT)

WHY LAG FEATURES?
"Yesterday's sales predict tomorrow's sales."

A lag of 7 days: what did this product sell 7 days ago?
This is the single most powerful feature in retail forecasting.

CRITICAL: We use groupby(['stockcode']) so lags are computed per product. Without this, it would mix one product's history with another product's — which is wrong!

In [9]:
print('\n[2/6] Lag features...')

GRP = ['stockcode']
for lag in [1, 2, 3, 7, 14, 21, 28]:
    df[f'demand_lag_{lag}d'] = df.groupby(GRP)['demand'].shift(lag)

df['price_lag_7d']    = df.groupby(GRP)['price'].shift(7)
df['price_change_7d'] = df['price'] - df['price_lag_7d'].fillna(df['price'])

print('9 lag features')


[2/6] Lag features...
9 lag features


## Rolling Window Features

WHY ROLLING AVERAGES?
A 7-day average smooths out day-to-day noise.
It tells the model: "what was the TREND this week?" rather than one noisy single-day value.

IMPORTANT: We use .shift(1) FIRST before rolling.
This prevents DATA LEAKAGE — using today's value to predict today (the model would be cheating!).

In [10]:
print('\n[3/6] Rolling features...')

shifted = df.groupby(GRP)['demand'].shift(1)

for window in [7, 14, 30]:
    df[f'demand_ma_{window}d'] = shifted.groupby(df['stockcode']).transform(
        lambda x: x.rolling(window, min_periods=1).mean())
    df[f'demand_std_{window}d'] = shifted.groupby(df['stockcode']).transform(
        lambda x: x.rolling(window, min_periods=1).std().fillna(0))

df['demand_max_7d'] = shifted.groupby(df['stockcode']).transform(
    lambda x: x.rolling(7, min_periods=1).max())
df['demand_min_7d'] = shifted.groupby(df['stockcode']).transform(
    lambda x: x.rolling(7, min_periods=1).min())

# Sum of past 7 days (total weekly demand)
df['demand_sum_7d'] = shifted.groupby(df['stockcode']).transform(
    lambda x: x.rolling(7, min_periods=1).sum())

print('11 rolling features')



[3/6] Rolling features...
11 rolling features


## Trend & Derived Features

In [11]:
print('\n[4/6] Trend features...')

df['demand_trend']   = df['demand_ma_7d'] - df['demand_ma_30d']
df['demand_wow_pct'] = (
    df.groupby(GRP)['demand']
    .pct_change(7)
    .replace([np.inf, -np.inf], 0)
    .fillna(0).clip(-1, 5)
)
df['demand_cv']         = df['demand_std_7d'] / (df['demand_ma_7d'] + 1)
df['log_demand']        = np.log1p(df['demand'])
df['expected_revenue']  = df['demand_ma_7d'] * df['price']

# FIX: count non-zero demand days in past 7 days
# Helps model know: "this product sells most days" vs "rarely"
df['nonzero_days_7d'] = shifted.groupby(df['stockcode']).transform(
    lambda x: (x > 0).rolling(7, min_periods=1).sum())

print('6 trend features')


[4/6] Trend features...
6 trend features


## Categorical Encoding

ML models only understand numbers, never text.

LabelEncoder converts: 'LAPTOP' → 0, 'PHONE' → 1, etc. We save the mapping so we can decode in the Streamlit app.

In [12]:
print('\n[5/6] Categorical encoding...')

le_prod    = LabelEncoder()
le_country = LabelEncoder()
df['product_enc'] = le_prod.fit_transform(df['stockcode'])
df['country_enc'] = le_country.fit_transform(df['country'])

product_map   = {str(c): int(i) for i,c in enumerate(le_prod.classes_)}
country_map   = {str(c): int(i) for i,c in enumerate(le_country.classes_)}
product_names = dict(zip(df['product_enc'].astype(str), df['description']))

print(f'{len(product_map)} products, {len(country_map)} countries encoded')


# FIX: HANDLE MISSING VALUES 
# lag cols are NaN for the first N rows of each product.
# Fill with that product's own mean (not 0!) where possible.
print('\n[6/6] Handling missing values (improved)...')


lag_cols = [c for c in df.columns if any(k in c for k in
    ['_lag_','_ma_','_std_','_max_','_min_','_sum_',
     'trend','_cv','wow','price_change','expected','nonzero'])]

before = df[lag_cols].isnull().sum().sum()

# Fill NaN lag values with each product's own rolling mean
# This is much better than filling with 0
for col in lag_cols:
    if df[col].isnull().any():
        # Fill with product-level median first
        product_medians = df.groupby('stockcode')[col].transform('median')
        df[col] = df[col].fillna(product_medians)
        # Any remaining NaN → 0
        df[col] = df[col].fillna(0)

after = df[lag_cols].isnull().sum().sum()
print(f'NaN values: {before:,} → {after}')


[5/6] Categorical encoding...
20 products, 1 countries encoded

[6/6] Handling missing values (improved)...
NaN values: 1,840 → 0


## Define Feature List

Lag/rolling features create NaN for the first N rows (because there are no past values yet).

We fill them with 0 — safe for tree-based models.

In [13]:
FEATURES = [
    # Time
    'year','month','day','day_of_week','week_of_year',
    'quarter','day_of_year',
    'month_sin','month_cos','dow_sin','dow_cos',
    'is_weekend','is_christmas_season','is_q4','is_january_dip',
    # Lag
    'demand_lag_1d','demand_lag_2d','demand_lag_3d',
    'demand_lag_7d','demand_lag_14d','demand_lag_21d','demand_lag_28d',
    # Rolling
    'demand_ma_7d','demand_ma_14d','demand_ma_30d',
    'demand_std_7d','demand_std_14d','demand_std_30d',
    'demand_max_7d','demand_min_7d','demand_sum_7d',
    # Trend
    'demand_trend','demand_wow_pct','demand_cv','nonzero_days_7d',
    # Price
    'price','price_lag_7d','price_change_7d','expected_revenue',
    # Categorical
    'product_enc','country_enc',
    # Context
    'num_invoices','num_customers',
]
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'demand'

print(f'\nTotal features : {len(FEATURES)}')
print(f'Target         : {TARGET}')
print(f'Dataset shape  : {df.shape}')


Total features : 43
Target         : demand
Dataset shape  : (22420, 55)


In [14]:
df.to_csv('C:/Users/HP/Downloads/edunet-IBM AIML project/retail_features.csv', index=False)
print('\nSaved: retail_features.csv')


Saved: retail_features.csv


In [15]:

config = {
    'features'     : FEATURES,
    'target'       : TARGET,
    'product_map'  : product_map,
    'country_map'  : country_map,
    'product_names': product_names,
}
with open('data/feature_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('\nretail_features.csv')
print('\nfeature_config.json')





retail_features.csv

feature_config.json
